# 00. Introduction: The Problem in Plain English

**Author:** Nikola Baburov &middot; **Module:** Individual Semester 6 &middot; **Date:** 2026-05-13

This notebook is the one place where I explain the problem from first principles. Notebooks 01 onwards assume the reader has been through this primer and skip the financial background.

**What I cover:**

1. What SPY is.
2. How to read a candlestick.
3. What Smart Money Concepts (SMC) means in plain language.
4. What a Fair Value Gap (FVG) is, geometrically.
5. Raw FVG vs Valid FVG side by side, with the extra criteria called out.
6. Why train a model at all when the labeller already exists.
7. Why this is a classification problem, not a price forecast.

Synthetic data is used throughout the figures so every shape stays clean and every criterion is labelled. One real SPY chart is included for context. No trading experience assumed.

## Setup

All constants live in the config cell. The two FVG labellers are imported from `src/` so the class-distribution chart later cannot drift from the production pipeline.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
pio.renderers.default = "notebook_connected"

from src.data.labels.fvg import FVGLabeller
from src.data.labels.valid_fvg import ValidFVGLabeller

SEED = 42
DATA_PATH = "../data/processed/spy_h1.parquet"

np.random.seed(SEED)
print(f"Seed set: {SEED}")

Seed set: 42


In [2]:
BULL = "#26a69a"
BEAR = "#ef5350"
TEXT = "#37474f"

def candle(sdf, x=None, showlegend=False):
    return go.Candlestick(
        x=x if x is not None else list(sdf.index),
        open=sdf["open"], high=sdf["high"], low=sdf["low"], close=sdf["close"],
        increasing_line_color=BULL, decreasing_line_color=BEAR, showlegend=showlegend,
    )

def style(fig, title, height=440):
    fig.update_layout(title=title, height=height, template="plotly_white",
                       xaxis_rangeslider_visible=False, yaxis_title="Price")
    fig.update_xaxes(type="category")
    return fig

def synth(opens, highs, lows, closes):
    return pd.DataFrame({"open": opens, "high": highs, "low": lows, "close": closes},
                        index=[f"t{i+1}" for i in range(len(opens))])

## 1. What Is SPY?

SPY is the ticker for the SPDR S&P 500 ETF Trust. One share is a tiny slice of the 500 largest US public companies bundled together. It is the most heavily traded equity instrument in the world.

Heavy trading matters here for one reason: clean, dense data. Almost no missing minutes, almost no weird price jumps. If a pattern is learnable anywhere, it is learnable on SPY. That is why I picked it for this MVP.

**Why hourly bars?** Daily bars are too coarse: a decade of SPY trading produces roughly 2,500 daily bars, which leaves too few labelled positives for a neural network to learn from. One-minute bars go the other way: microstructure noise (bid-ask bounce, partial fills, stale quotes) dominates over structural patterns, and the three-bar FVG geometry fires so frequently it becomes trivial to satisfy. Hourly bars are the practical middle ground. The dataset covers 17,591 H1 bars over ten years, enough for gradient-based learning, and the signal-to-noise ratio is high enough that an FVG label at this resolution carries genuine information.

## 2. Reading a Candlestick

Each bar in this dataset summarises one hour of trading with five numbers (OHLCV):

- **Open**: first traded price of the hour
- **High**: highest traded price of the hour
- **Low**: lowest traded price of the hour
- **Close**: last traded price of the hour
- **Volume**: total shares traded in the hour

A candlestick is a visual shorthand for those five numbers:

- The thick rectangle is the **body**, spanning open to close.
- The thin lines above and below are the **wicks**, spanning low and high.
- **Green** body: close above open (price rose during the hour).
- **Red** body: close below open (price fell during the hour).

Figure 1 labels every part on a synthetic two-bar sequence with arrows.

In [3]:
demo = synth([100, 106], [108, 108], [98, 100], [105, 101])
demo.index = ["green hour", "red hour"]

fig = go.Figure(candle(demo))

# Green bar annotations
fig.add_annotation(x="green hour", y=demo["high"].iloc[0], ax=120, ay=-50,
                   text="<b>Upper wick</b><br>reaches the high", showarrow=True, arrowhead=2)
fig.add_annotation(x="green hour", y=demo["low"].iloc[0], ax=120, ay=50,
                   text="<b>Lower wick</b><br>reaches the low", showarrow=True, arrowhead=2)
fig.add_annotation(x="green hour", y=(demo["open"].iloc[0]+demo["close"].iloc[0])/2, ax=170, ay=0,
                   text="<b>Green body</b><br>open=100, close=105 (price rose)", showarrow=True, arrowhead=2)

# Red bar annotation
fig.add_annotation(x="red hour", y=(demo["open"].iloc[1]+demo["close"].iloc[1])/2, ax=170, ay=0,
                   text="<b>Red body</b><br>open=106, close=101 (price fell)", showarrow=True, arrowhead=2)

style(fig, "Figure 1. Anatomy of a candlestick. Body spans open to close, wicks reach the high and the low.", height=460).show()

In [4]:
df = pd.read_parquet(DATA_PATH)
ohlcv = df[["open", "high", "low", "close", "volume"]]
sample = ohlcv.iloc[200:210]
fig = go.Figure(candle(sample, x=sample.index.strftime("%m-%d %H:%M")))
style(fig,
    f"Figure 2. Ten real SPY hourly bars starting {sample.index[0].date()}. <br>"
    "<sub>Source: Alpaca Markets, raw adjustment, RTH only (09:30 to 15:59 ET). "
    "X-axis is categorical (consecutive trading hours, no overnight or weekend gaps).</sub>").show()
print(f"Loaded {len(df):,} hourly bars, range {df.index.min().date()} to {df.index.max().date()}")

Loaded 17,591 hourly bars, range 2016-01-04 to 2025-12-30


Figures 1 and 2 show the entire data type every model in this project consumes: a stream of five numbers per hour, nothing else. No news headlines, no fundamentals, no order book. The whole modelling question is whether structural patterns hidden in this thin signal are recoverable.

## 3. Smart Money Concepts (SMC) in One Paragraph

Smart Money Concepts is a school of technical analysis. Its core claim is that large institutional traders leave detectable footprints on price charts because they cannot enter or exit a position all at once without moving the market against themselves. Whether or not the underlying market theory is true is irrelevant for this project. The structures SMC describes (order blocks, breaks of structure, liquidity sweeps, fair value gaps) are geometrically well defined. That makes them labellable, which makes them suitable targets for supervised learning. The thesis is narrower than the trading philosophy:

> Can a neural network learn to recognise SMC structures from raw OHLCV alone, well enough to match a rule-based labeller?

Of the SMC structures I picked the **Fair Value Gap** for the MVP. Simplest geometric definition (three candles, two inequalities), and most common, which matters when class imbalance is already extreme. Order blocks, breaks of structure, and liquidity sweeps are all explicitly out of scope for this MVP.

## 4. What Is a Fair Value Gap (FVG)?

Picture price walking up a staircase one step at a time. Most of the time it touches every step. Occasionally it leaps over a step, leaving a visible gap in the air where price never traded. SMC calls that gap a **Fair Value Gap**. The folklore is that price tends to return and "fill" the gap later.

Geometrically it is a three-candle pattern. Call them N-1, N, N+1.

**Bullish FVG** (price gapped up): high of N-1 is below low of N+1, and the body of N is green.

**Bearish FVG** (price gapped down): low of N-1 is above high of N+1, and the body of N is red.

Figure 3 illustrates both shapes with the gap zone shaded and each candle labelled.

In [5]:
bull = synth([100, 101, 106], [102, 105, 110], [99,  100, 105], [101, 104, 109])
bear = synth([110, 109, 104], [111, 110, 106], [108, 105, 100], [109, 106, 101])

fig = make_subplots(rows=1, cols=2,
                     subplot_titles=("Bullish FVG (price gapped up)", "Bearish FVG (price gapped down)"))
for col, sdf in enumerate([bull, bear], start=1):
    fig.add_trace(candle(sdf), row=1, col=col)

# Bullish gap
y0b, y1b = bull["high"].iloc[0], bull["low"].iloc[2]
fig.add_shape(type="rect", x0="t1", x1="t3", y0=y0b, y1=y1b,
              fillcolor=BULL, opacity=0.22, line=dict(color=BULL, width=1, dash="dot"), row=1, col=1)
fig.add_annotation(x="t3", y=(y0b+y1b)/2, ax=80, ay=0,
                   text="<b>Gap zone</b><br>price skipped over this range",
                   showarrow=True, arrowhead=2, font=dict(color="#1b5e20"), row=1, col=1)
for i, lbl in enumerate(["N-1", "N (bullish body)", "N+1"]):
    fig.add_annotation(x=f"t{i+1}", y=bull["low"].iloc[i] - 1.2, text=lbl, showarrow=False,
                       font=dict(size=11, color=TEXT), row=1, col=1)

# Bearish gap
y0r, y1r = bear["high"].iloc[2], bear["low"].iloc[0]
fig.add_shape(type="rect", x0="t1", x1="t3", y0=y0r, y1=y1r,
              fillcolor=BEAR, opacity=0.22, line=dict(color=BEAR, width=1, dash="dot"), row=1, col=2)
fig.add_annotation(x="t3", y=(y0r+y1r)/2, ax=80, ay=0,
                   text="<b>Gap zone</b><br>price skipped over this range",
                   showarrow=True, arrowhead=2, font=dict(color="#b71c1c"), row=1, col=2)
for i, lbl in enumerate(["N-1", "N (bearish body)", "N+1"]):
    fig.add_annotation(x=f"t{i+1}", y=bear["high"].iloc[i] + 1.2, text=lbl, showarrow=False,
                       font=dict(size=11, color=TEXT), row=1, col=2)

fig.update_layout(title="Figure 3. Bullish and bearish FVG geometry. Shaded rectangle = untraded gap zone.",
                   height=480, template="plotly_white")
fig.update_xaxes(type="category", rangeslider_visible=False)
fig.show()

The shaded rectangle is the price range that price skipped over. That rectangle is what the labeller marks, and what the model is asked to recognise from the surrounding 60 hours of context.

Label placement: the label sits at candle **N+1**, not N. The three-candle pattern only completes when N+1 closes, so labelling at N would require future information. Placing the label at N+1 keeps the dataset honest. A pytest fixture enforces this rule across the whole pipeline.

## 5. Raw FVG vs Valid FVG, Side by Side

The geometric definition above is what I call the **raw FVG**. It fires on roughly **24%** of SPY hourly bars, because the bar is very low: a gap plus a coloured body. Most of those firings sit inside news spikes or lunch-hour drift, geometrically valid but tradingly meaningless.

Practitioners filter raw FVGs with the **six TradingLab criteria**:

1. Geometric FVG exists.
2. The next candle (N+2) reacts by closing back inside the gap zone.
3. A support/resistance pivot coincides with the gap zone.
4. All passing candidates labelled (no priority elimination).
5. The gap sits on the correct side of the recent 50-bar swing midpoint.
6. A Break of Structure occurred recently before the gap.

Only ~3.2% of bars pass all six. That is the **Valid FVG** target (`fvg_valid`). It is what I train on. The label moves to N+2 because criterion 2 needs the reaction candle to close.

Figure 4 shows both labellers on the same synthetic setup. **Left panel:** raw fires, Valid does not. **Right panel:** raw fires *and* Valid fires, with the extra criteria pointed out by arrows.

In [6]:
# Left panel: raw passes (geometric gap), Valid fails (no reaction at N+2)
# Bars: t1=N-1, t2=N (bullish), t3=N+1 (gap above N-1.high), t4=N+2 (no reaction, keeps going up)
raw_only = synth(
    opens =[100, 101, 106, 110],
    highs =[102, 105, 110, 113],
    lows  =[99,  100, 105, 109],
    closes=[101, 104, 109, 112],
)

# Right panel: raw + Valid both pass
# Bars: t0=BOS marker bar, t1=N-1, t2=N (bullish), t3=N+1, t4=N+2 reaction closes inside gap
valid_ok = synth(
    opens =[95,  100, 101, 106, 108],
    highs =[101, 102, 105, 110, 110],
    lows  =[94,  99,  100, 105, 104],
    closes=[100, 101, 104, 109, 105],   # t4 close=105, inside the gap (102..105)
)
valid_ok.index = ["BOS", "N-1", "N", "N+1", "N+2"]
raw_only.index = ["N-1", "N", "N+1", "N+2"]

fig = make_subplots(rows=1, cols=2,
                     subplot_titles=("Raw fires only (Valid rejects: no reaction)",
                                     "Raw + Valid both fire (all 6 criteria pass)"))
fig.add_trace(candle(raw_only), row=1, col=1)
fig.add_trace(candle(valid_ok), row=1, col=2)

# Left: gap zone
y0L, y1L = raw_only["high"].iloc[0], raw_only["low"].iloc[2]
fig.add_shape(type="rect", x0="N-1", x1="N+2", y0=y0L, y1=y1L,
              fillcolor=BULL, opacity=0.18, line=dict(color=BULL, width=1, dash="dot"), row=1, col=1)
fig.add_annotation(x="N+1", y=(y0L+y1L)/2, text="<b>Gap zone</b>", showarrow=False,
                   font=dict(color="#1b5e20"), row=1, col=1)
fig.add_annotation(x="N+2", y=raw_only["close"].iloc[3], ax=-60, ay=-50,
                   text="<b>N+2 keeps rising</b><br>does NOT close inside gap<br>Valid rejects (criterion 2 fails)",
                   showarrow=True, arrowhead=2, font=dict(color="#b71c1c"), row=1, col=1)
fig.add_annotation(x="N", y=raw_only["high"].iloc[1] + 1.5,
                   text="Raw checks: gap + bullish body → ✓",
                   showarrow=False, font=dict(size=10, color=TEXT), row=1, col=1)

# Right: gap zone
y0R, y1R = valid_ok["high"].iloc[1], valid_ok["low"].iloc[3]
fig.add_shape(type="rect", x0="BOS", x1="N+2", y0=y0R, y1=y1R,
              fillcolor=BULL, opacity=0.18, line=dict(color=BULL, width=1, dash="dot"), row=1, col=2)
fig.add_annotation(x="N+1", y=(y0R+y1R)/2, text="<b>Gap zone</b>", showarrow=False,
                   font=dict(color="#1b5e20"), row=1, col=2)
fig.add_annotation(x="BOS", y=valid_ok["high"].iloc[0], ax=0, ay=-50,
                   text="<b>Prior BOS</b><br>(criterion 6 ✓)",
                   showarrow=True, arrowhead=2, font=dict(color=TEXT), row=1, col=2)
fig.add_annotation(x="N+2", y=valid_ok["close"].iloc[4], ax=60, ay=-50,
                   text="<b>N+2 reacts</b><br>close back inside gap<br>(criterion 2 ✓)",
                   showarrow=True, arrowhead=2, font=dict(color="#1b5e20"), row=1, col=2)
fig.add_annotation(x="N", y=valid_ok["high"].iloc[2] + 1.5,
                   text="Raw + Valid both ✓",
                   showarrow=False, font=dict(size=10, color=TEXT), row=1, col=2)

fig.update_layout(title="Figure 4. Raw FVG vs Valid FVG on synthetic data. Same geometric gap, different verdicts.",
                   height=520, template="plotly_white")
fig.update_xaxes(type="category", rangeslider_visible=False)
fig.update_yaxes(title_text="Price")
fig.show()

Figure 4 makes the difference concrete. The geometric ingredient (gap between N-1 high and N+1 low, bullish body at N) is identical on both sides. The split happens at N+2. On the left, price keeps marching up: criterion 2 fails, Valid stays silent. On the right, N+2 closes back inside the gap zone (the "reaction" candle), there is a prior break of structure, and the Valid labeller fires. The other four criteria (pivot coincidence, swing-midpoint side, candidate priority, geometric pattern) are also checked under the hood; criteria 2 and 6 are the two that visibly differentiate cases in this plot.

Figure 5 quantifies what filtering does to the dataset as a whole. The shift between the two targets is the single most important design decision in the project.

In [7]:
raw_labels = FVGLabeller().label(ohlcv)
valid_labels = ValidFVGLabeller().label(ohlcv)

def counts(labels):
    return [(labels == 0).sum(), (labels == 1).sum(), (labels == -1).sum()]

raw_c, val_c = counts(raw_labels), counts(valid_labels)
names = ["none", "bullish", "bearish"]
total = len(ohlcv)

print(f"Total bars: {total:,}\n")
print(f"{'class':<10}{'raw FVG':>15}{'%':>10}{'Valid FVG':>15}{'%':>10}")
for n, r, v in zip(names, raw_c, val_c):
    print(f"{n:<10}{r:>15,}{100*r/total:>9.2f}%{v:>15,}{100*v/total:>9.2f}%")

fig = go.Figure()
fig.add_trace(go.Bar(name="Raw FVG", x=names, y=[100*c/total for c in raw_c], marker_color="#90a4ae",
                     text=[f"{100*c/total:.1f}%" for c in raw_c], textposition="outside"))
fig.add_trace(go.Bar(name="Valid FVG", x=names, y=[100*c/total for c in val_c], marker_color="#1e88e5",
                     text=[f"{100*c/total:.1f}%" for c in val_c], textposition="outside"))
fig.update_layout(
    title="Figure 5. Class distribution: raw FVG vs Valid FVG on SPY H1 2016 to 2025. <br>"
          "<sub>Raw fires on ~24% of bars. Valid survives at ~3%. Valid is the canonical training target.</sub>",
    yaxis_title="Share of bars (%)", barmode="group", template="plotly_white", height=440,
)
fig.show()

Total bars: 17,591

class             raw FVG         %      Valid FVG         %
none               13,239    75.26%         17,020    96.75%
bullish             2,595    14.75%            324     1.84%
bearish             1,757     9.99%            247     1.40%


Figure 5 shows the trade-off. The raw labeller hands the model many positives, but most are spurious. The valid labeller delivers a much cleaner signal at the cost of a severe class-imbalance regime. I take the cleaner signal and pay the imbalance cost, because a model that learns a noisy target well is still useless. Section 6 covers how I handle the imbalance.

## 6. Why Train a Model When the Labeller Already Exists?

Fair question. The Valid FVG labeller is a deterministic algorithm. Given a window of OHLCV bars it returns the correct label with zero error. If the goal is just to mark FVGs on a chart, no neural network is needed. Why train one?

Five reasons, in order of how much each one actually matters for this project.

**1. Inference cost.** The labeller has to compute a 50-bar swing midpoint, a Break-of-Structure check with its own lookback, ATR-expanded pivot zones, and the six-criteria gate per candidate bar. That is heavy preprocessing on every new bar. A trained network is one forward pass over the 60-bar window, roughly a few thousand multiply-accumulate operations. For live annotation this matters.

**2. Calibrated confidence.** The labeller emits hard `0`, `1`, or `-1`. A model emits a probability vector. That probability is what makes overlay-on-threshold possible: I can ship a high-precision overlay (threshold 0.9) for users who want few false positives, or a high-recall overlay (threshold 0.5) for users who want every plausible gap flagged. The labeller cannot produce that knob.

**3. Robustness at the edges.** The labeller is brittle by construction: small parameter changes (the BOS lookback, the pivot ATR multiplier, the swing window) cascade through the six criteria. A network trained against the labeller's output smooths over noise in those edges and degrades gracefully on bars where one criterion barely fails or barely passes.

**4. Transferability.** The same network architecture retrained on QQQ or AAPL costs an afternoon. Porting the labeller to a new instrument means re-tuning the BOS lookback, the pivot ATR, and the swing window for that instrument's volatility regime. The network learns a shared representation; the rule-based labeller does not.

**5. It is the assignment.** This module is a deep-learning course. Reimplementing the labeller in production would be the right engineering answer; demonstrating ML competence requires training a model and reporting its F1 against the rule-based oracle. The labeller is the target the network has to match, not the deliverable.

The honest summary: in a deployment where compute is cheap and FVG is the only target, the labeller wins. In a deployment where confidence calibration, multi-instrument support, or graceful degradation matters, the model wins. Here, the model is the deliverable both for academic reasons and for the calibration and transfer points above.

## 7. Classification, Not Forecasting

A common misread of this project is that I am predicting future prices. I am not. The task is:

> Given the last 60 hours of OHLCV ending at time *t*, decide whether bar *t* is a bullish FVG, a bearish FVG, or neither.

Three-class classification. Output is a label plus a confidence score, then a Plotly overlay rectangle on the chart. No price target, no direction call, no trade signal. The model is an annotator, not a forecaster. That framing matters because it changes:

- **Metric.** Macro F1, not RMSE or accuracy.
- **Validation.** Temporal hold-out, not random k-fold.
- **Deployment.** Live chart overlay, not a buy/sell trigger.

Two constraints follow and shape every downstream notebook.

**Class imbalance.** The positive classes together are ~3% of bars. A model that predicts "none" on every bar reaches 97% accuracy and learns nothing. Accuracy is therefore banned as the primary metric. I use **macro F1** (the unweighted mean of the three per-class F1 scores) and report per-class precision and recall in every evaluation. To push the model toward the rare classes during training I use inverse-frequency-weighted cross-entropy, with weights persisted in `data/processed/class_weights.json` at split time.

**Temporal split.** Time series cannot be shuffled. A random 80/20 split would let the model train on bars from 2024 and test on bars from 2017, leaking future regimes into the past. I split chronologically: 2016 to 2021 for training, 2022 for validation, 2023 to 2025 for the final test set. Boundaries live in `src/data/split.py`. The split is enforced by a pytest fixture and never crossed during hyperparameter tuning.

In [8]:
# Figure 6: realistic Valid FVG annotation - cleaner, less cluttered.
# 15-bar synthetic SPY-style sequence containing a clean Valid FVG.
# Label sits at N+2 (matches ValidFVGLabeller offset).

mock = pd.DataFrame({
    "open":  [420.0, 419.2, 421.0, 420.4, 421.8, 423.0, 422.2, 421.4, 422.0, 424.6, 425.2, 425.0, 423.7, 424.0, 424.8],
    "high":  [420.6, 421.3, 421.4, 422.1, 423.4, 423.6, 422.6, 421.9, 424.4, 425.5, 425.6, 425.4, 424.4, 424.9, 425.6],
    "low":   [418.9, 418.8, 420.2, 420.1, 421.6, 421.9, 421.3, 421.0, 421.9, 424.5, 424.7, 423.4, 423.3, 423.5, 424.2],
    "close": [419.2, 421.0, 420.4, 421.8, 423.0, 422.2, 421.4, 421.6, 424.3, 425.2, 425.0, 423.7, 424.0, 424.8, 425.4],
}, index=[f"t{i+1}" for i in range(15)])

N_MINUS_1, N, N_PLUS_1, N_PLUS_2 = "t8", "t9", "t10", "t11"
gap_bottom = mock.loc[N_MINUS_1, "high"]
gap_top    = mock.loc[N_PLUS_1,  "low"]

fig = go.Figure(candle(mock))

# Gap zone rectangle
fig.add_shape(
    type="rect", x0=N_MINUS_1, x1=N_PLUS_2,
    y0=gap_bottom, y1=gap_top,
    fillcolor=BULL, opacity=0.30,
    line=dict(color=BULL, width=1.2, dash="dot"),
)

# Single model-output callout (the headline, only annotation that sits on the chart)
fig.add_annotation(
    x=N_PLUS_2, y=gap_top + 0.7,
    ax=90, ay=-40,
    text="<b>Model: bullish Valid FVG</b><br>confidence 0.81",
    showarrow=True, arrowhead=2,
    arrowcolor=BULL,
    font=dict(color="#1b5e20", size=12),
    bgcolor="white", bordercolor=BULL, borderwidth=1.5,
)

# Compact N-1 / N / N+1 / N+2 ticks under bars (one short label each)
for bar_id, lbl in [(N_MINUS_1, "N-1"), (N, "N"), (N_PLUS_1, "N+1"), (N_PLUS_2, "N+2")]:
    fig.add_annotation(
        x=bar_id, y=mock["low"].min() - 0.5, text=lbl,
        showarrow=False, font=dict(size=10, color=TEXT),
    )

style(
    fig,
    "Figure 6. Mock model output on a synthetic Valid FVG.<br>"
    "<sub>Green rectangle = predicted gap zone. Label placed at N+2 (ValidFVGLabeller offset).</sub>",
    height=440,
).show()


Figure 6 is what the deployment view looks like: one shaded gap rectangle, one labelled prediction, one confidence number. The three-candle FVG geometry sits at N-1 / N / N+1; the label is emitted at N+2 because that is the earliest bar where every Valid FVG criterion (gap, reaction, BOS, pivot, swing side) is knowable. In production the overlay only renders when confidence clears a tuned threshold; everything else passes through unmarked.


## Final Summary

**What this notebook is.** A standalone primer that explains the problem, the data type, the target, and the framing once, so subsequent notebooks can focus on modelling.

**Concrete facts established here:**

- Dataset: SPY hourly bars from Alpaca, 2016-01-04 to 2025-12-30, 17,591 rows, OHLCV only, RTH only.
- Target: ternary classification (none / bullish FVG / bearish FVG).
- Canonical labeller: `ValidFVGLabeller` (`fvg_valid`), label index N+2, positive rate ~3.2%.
- Historical labeller: `FVGLabeller` (`fvg_raw`), label index N+1, positive rate ~24%, kept for legacy comparisons only.
- Primary metric: macro F1. Accuracy explicitly rejected because of the imbalance.
- Validation strategy: temporal split 2016 to 2021 / 2022 / 2023 to 2025. Never shuffled.
- Output shape: per-bar label plus confidence, rendered as a Plotly chart overlay. Not a price prediction.

**What you can do now.**

- Read any candlestick chart in this repo: open, high, low, close body and wicks, green means close above open, red means close below open.
- Recognise an FVG geometrically and explain raw vs Valid in one breath: raw is the three-candle gap alone (~24% of bars); Valid adds five more criteria including a reaction candle and a prior break of structure, cutting positives to ~3%.
- Explain why macro F1 is the metric and why the split is chronological: accuracy collapses to 97% by predicting "none" every bar, and shuffling a time series leaks future market regimes into past training data.

**Limitations of this primer.** I only cover FVG. SMC contains several other structures (order blocks, breaks of structure, liquidity sweeps) which are out of scope for the MVP. I also do not justify the trading philosophy behind SMC; this project tests whether the structures are learnable, not whether trading on them is profitable.

**LO mapping.**

- **LO1 (Data Preparation).** Data provenance is named (Alpaca free tier, raw adjustment, RTH only), schema is shown, and the resampling decision (`TimeFrame.Minute` + RTH filter + anchored H1 resample) is referenced.
- **LO2 (Evaluation choice).** Macro F1 is justified against accuracy with a concrete numeric argument (97% accuracy floor from the imbalance).
- **LO3 (Methodology).** Picking Valid FVG over raw FVG is stated with the trade-off (cleaner signal, harder imbalance) and the criterion that decided it.
- **LO4 (Reflection).** The framing shift from price prediction to chart annotation is acknowledged as the single most important conceptual move in the project, because it changed every downstream choice.

**References.**

- TradingLab. (2024). *Valid Fair Value Gap detection: a six-criteria rule.* Community documentation. Retrieved 2026-05-08.
- Alpaca Markets. (2026). *Historical market data API, Stocks v2.* https://docs.alpaca.markets/. Retrieved 2026-05-08.
- ICT / Inner Circle Trader. (2022). *Smart Money Concepts, Fair Value Gap.* Retrieved 2026-05-08.